In [0]:
-- ============================================
-- VIEW 1: Segment classification (who's eligible, by type)
-- ============================================
CREATE OR REPLACE TEMP VIEW bf25_member_segments AS
WITH
first_timers AS (
    SELECT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND cc_user_profile_id IS NOT NULL
    GROUP BY cc_user_profile_id
    HAVING COUNT(*) = 1
),
upgrades_took_effect AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_first_time AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    INNER JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_returning AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN first_timers AS f ON e.cc_user_profile_id = f.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_total_signups AS (
    SELECT DISTINCT e.cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
    LEFT ANTI JOIN upgrades_took_effect AS ute ON e.cc_user_profile_id = ute.cc_user_profile_id
    WHERE e.event_type = 'subscription_created' AND e.plan_id = 'annual1'
      AND e.reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
a_exits AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'annual1' AND cc_user_profile_id IS NOT NULL
      AND changed IS NOT TRUE
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_direct_signup_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_created' AND plan_id = 'annual1' AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_upgrade_starts AS (
    SELECT cc_user_profile_id, MIN(CAST(reporting_datetime AS DATE)) AS start_date
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'subscription_cancelled' AND plan_id = 'monthly1' AND changed = TRUE AND reporting_datetime IS NOT NULL
    GROUP BY cc_user_profile_id
),
annual_starts AS (
    SELECT cc_user_profile_id, MIN(start_date) AS start_date
    FROM (SELECT * FROM annual_direct_signup_starts UNION ALL SELECT * FROM annual_upgrade_starts)
    GROUP BY cc_user_profile_id
),
annual_payments_window AS (
    SELECT DISTINCT cc_user_profile_id, reporting_datetime
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type = 'payment_succeeded' AND plan_id = 'annual1'
      AND reporting_datetime BETWEEN '2025-11-28' AND '2025-12-01'
),
annual_renewed AS (
    SELECT DISTINCT p.cc_user_profile_id
    FROM annual_payments_window AS p
    INNER JOIN annual_starts AS s ON p.cc_user_profile_id = s.cc_user_profile_id
    LEFT ANTI JOIN a_total_signups AS sg ON sg.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN a_exits AS ex ON ex.cc_user_profile_id = p.cc_user_profile_id
    LEFT ANTI JOIN upgrades_took_effect AS ute ON ute.cc_user_profile_id = p.cc_user_profile_id
    WHERE DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date) >= 360
      AND (MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) <= 5
           OR MOD(DATEDIFF(CAST(p.reporting_datetime AS DATE), s.start_date), 365) >= 360)
),
prev_window_snapshot AS (
    SELECT DISTINCT cc_user_profile_id, plan_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id = '20251127'
),
g_purchasers AS (
    SELECT DISTINCT purchaser_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE CAST(from_utc_timestamp(CAST(purchase_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
g_redeemers AS (
    SELECT DISTINCT redeemer_id
    FROM cpx_dataanalytics_silver.cineclub.gifting
    WHERE is_redeemed = TRUE
      AND CAST(from_utc_timestamp(CAST(redemption_date AS TIMESTAMP), 'America/New_York') AS DATE)
          BETWEEN '2025-11-28' AND '2025-12-01'
),
gift_purchasers_net_new AS (
    SELECT t.purchaser_id AS cc_user_profile_id FROM g_purchasers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.purchaser_id = p.cc_user_profile_id
),
gift_redeemers_net_new AS (
    SELECT t.redeemer_id AS cc_user_profile_id FROM g_redeemers t
    LEFT ANTI JOIN prev_window_snapshot p ON t.redeemer_id = p.cc_user_profile_id
)
SELECT cc_user_profile_id, 'New' AS segment, 'CPA' AS cost_bucket FROM a_first_time
UNION ALL
SELECT cc_user_profile_id, 'Returning', 'CPA' FROM a_returning
UNION ALL
SELECT cc_user_profile_id, 'Upgraded', 'CPA' FROM upgrades_took_effect
UNION ALL
SELECT cc_user_profile_id, 'Gift_Purchaser_New', 'CPA' FROM gift_purchasers_net_new
UNION ALL
SELECT cc_user_profile_id, 'Gift_Redeemer_New', 'CPA' FROM gift_redeemers_net_new
UNION ALL
SELECT cc_user_profile_id, 'Renewed', 'CPR' FROM annual_renewed;


-- ============================================
-- VIEW 2: cc_user_profile_id -> cde_id lookup
-- ============================================
CREATE OR REPLACE TEMP VIEW bf25_cde_lookup AS
WITH cde_lookup_window AS (
    SELECT
        cc_user_profile_id,
        cde_id,
        ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY snapshot_date_id DESC) AS rn
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id BETWEEN '20251127' AND '20251201'
)
SELECT cc_user_profile_id, cde_id
FROM cde_lookup_window
WHERE rn = 1;


-- ============================================
-- VIEW 3: Valid voucher redemptions (BIN 7146, Jan-Mar 2026, deduped, capped at 3)
-- ============================================
CREATE OR REPLACE TEMP VIEW bf25_redemptions_per_member AS
WITH
voucher_trans AS (
    SELECT vt.*, vd.*
    FROM cpx_dataanalytics_gold.marketing.mkt_tbltransvoucher AS vt
    LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_tblvouchertype AS vd
        ON vt.voucher_type_id = vd.voucher_type_id
    WHERE vd.n_voucher_code LIKE '7146%'
),
voucher_trans_joined AS (
    SELECT
        v1.*,
        v3.CDE_ID,
        v3.LocationId,
        v3.Transaction_Number,
        v3.VisitDateId,
        CASE WHEN v3.Transaction_Number IS NOT NULL THEN 1 ELSE 0 END AS Transaction_Matched_Flag
    FROM voucher_trans AS v1
    LEFT JOIN cpx_dataanalytics_gold.customer360.c360_f_transaction AS v3
        ON v1.l_location_id = v3.LocationId
       AND v1.trans_c_lgn_number = v3.Transaction_Number
       AND date_format(v1.session_dtm_realshow, 'yyyyMMdd') = v3.VisitDateId
    WHERE v3.VisitDateId BETWEEN '20260101' AND '20260331'
)
SELECT
    CDE_ID,
    COUNT(DISTINCT CONCAT(Transaction_Number, '-', VisitDateId, '-', LocationId)) AS redemption_count
FROM voucher_trans_joined
WHERE Transaction_Matched_Flag = 1
GROUP BY CDE_ID
HAVING COUNT(DISTINCT CONCAT(Transaction_Number, '-', VisitDateId, '-', LocationId)) <= 3;


-- ============================================
-- VIEW 4: Segment sizes with directly-matched redemptions (attribution reference only)
-- ============================================
CREATE OR REPLACE TEMP VIEW bf25_segment_summary AS
SELECT
    ms.segment,
    ms.cost_bucket,
    COUNT(DISTINCT ms.cc_user_profile_id) AS segment_size,
    COUNT(DISTINCT rpm.CDE_ID) AS directly_matched_redeemers,
    COALESCE(SUM(rpm.redemption_count), 0) AS directly_matched_redemptions
FROM bf25_member_segments ms
LEFT JOIN bf25_cde_lookup cl ON ms.cc_user_profile_id = cl.cc_user_profile_id
LEFT JOIN bf25_redemptions_per_member rpm ON CAST(cl.cde_id AS STRING) = CAST(rpm.CDE_ID AS STRING)
GROUP BY ms.segment, ms.cost_bucket;


-- ============================================
-- VIEW 5: Proportional redemption allocation by segment
-- (voucher can be passed along, so direct match undercounts ~48%;
--  allocate total valid redemptions proportionally by segment size within CPA/CPR)
-- ============================================
CREATE OR REPLACE TEMP VIEW bf25_acquisition_breakdown AS
WITH
segment_sizes AS (
    SELECT segment, cost_bucket, segment_size
    FROM bf25_segment_summary
),
bucket_totals AS (
    SELECT cost_bucket, SUM(segment_size) AS bucket_size
    FROM segment_sizes
    GROUP BY cost_bucket
),
grand_total AS (
    SELECT SUM(bucket_size) AS total_population FROM bucket_totals
),
total_valid_redemptions AS (
    SELECT SUM(redemption_count) AS total_redemptions
    FROM bf25_redemptions_per_member
),
-- Stage 1: split total redemptions between CPA and CPR by population share
bucket_redemptions AS (
    SELECT
        bt.cost_bucket,
        bt.bucket_size,
        ROUND(bt.bucket_size * 1.0 / gt.total_population * tvr.total_redemptions, 1) AS bucket_allocated_redemptions
    FROM bucket_totals bt
    CROSS JOIN grand_total gt
    CROSS JOIN total_valid_redemptions tvr
)
-- Stage 2: split each bucket's redemptions across its segments by segment share
SELECT
    ss.segment,
    ss.cost_bucket,
    ss.segment_size,
    br.bucket_size,
    ROUND(ss.segment_size * 1.0 / br.bucket_size, 4) AS share_of_bucket,
    ROUND(ss.segment_size * 1.0 / br.bucket_size * br.bucket_allocated_redemptions, 1) AS allocated_redemptions
FROM segment_sizes ss
JOIN bucket_redemptions br ON ss.cost_bucket = br.cost_bucket
ORDER BY ss.cost_bucket, ss.segment;

-- View 4: Segment sizes with directly-matched redemptions (undercounted reference)
SELECT * FROM bf25_segment_summary;

-- View 5: Final acquisition/renewal breakdown with proportional redemption allocation
SELECT * FROM bf25_acquisition_breakdown;

WITH
signups_by_segment AS (
    SELECT
        segment,
        cost_bucket,
        COUNT(DISTINCT cc_user_profile_id) AS signups,
        COUNT(DISTINCT cc_user_profile_id) AS vouchers_issued
    FROM bf25_member_segments
    GROUP BY segment, cost_bucket
),
redeemed_by_segment AS (
    SELECT
        ms.segment,
        COUNT(DISTINCT rpm.CDE_ID) AS members_redeemed,
        COALESCE(SUM(rpm.redemption_count), 0) AS vouchers_redeemed
    FROM bf25_member_segments ms
    LEFT JOIN bf25_cde_lookup cl ON ms.cc_user_profile_id = cl.cc_user_profile_id
    LEFT JOIN bf25_redemptions_per_member rpm ON CAST(cl.cde_id AS STRING) = CAST(rpm.CDE_ID AS STRING)
    GROUP BY ms.segment
),
combined AS (
    SELECT
        sbs.segment,
        sbs.cost_bucket,
        sbs.signups,
        sbs.vouchers_issued,
        rbs.members_redeemed,
        rbs.vouchers_redeemed
    FROM signups_by_segment sbs
    LEFT JOIN redeemed_by_segment rbs ON sbs.segment = rbs.segment
),
correction_factor AS (
    -- true total valid redemptions / total directly-matched redemptions
    SELECT
        (SELECT SUM(redemption_count) FROM bf25_redemptions_per_member) * 1.0
        / (SELECT SUM(vouchers_redeemed) FROM combined) AS factor
)
SELECT
    c.segment,
    c.cost_bucket,
    c.signups,
    c.vouchers_issued,
    c.members_redeemed,
    c.vouchers_redeemed,
    ROUND(c.vouchers_redeemed * cf.factor, 1) AS scaled_redemptions,
    ROUND(c.vouchers_redeemed * 1.0 / c.vouchers_issued, 4) AS redemption_rate_direct,
    ROUND(c.vouchers_redeemed * cf.factor / c.vouchers_issued, 4) AS redemption_rate_scaled
FROM combined c
CROSS JOIN correction_factor cf

UNION ALL

SELECT
    'TOTAL',
    NULL,
    SUM(c.signups),
    SUM(c.vouchers_issued),
    SUM(c.members_redeemed),
    SUM(c.vouchers_redeemed),
    ROUND(SUM(c.vouchers_redeemed * cf.factor), 1),
    ROUND(SUM(c.vouchers_redeemed) * 1.0 / SUM(c.vouchers_issued), 4),
    ROUND(SUM(c.vouchers_redeemed * cf.factor) / SUM(c.vouchers_issued), 4)
FROM combined c
CROSS JOIN correction_factor cf

ORDER BY cost_bucket, segment;

WITH
signups_by_segment AS (
    SELECT
        segment,
        cost_bucket,
        COUNT(DISTINCT cc_user_profile_id) AS signups
    FROM bf25_member_segments
    GROUP BY segment, cost_bucket
),
redeemed_by_segment AS (
    SELECT
        ms.segment,
        COALESCE(SUM(rpm.redemption_count), 0) AS vouchers_redeemed
    FROM bf25_member_segments ms
    LEFT JOIN bf25_cde_lookup cl ON ms.cc_user_profile_id = cl.cc_user_profile_id
    LEFT JOIN bf25_redemptions_per_member rpm ON CAST(cl.cde_id AS STRING) = CAST(rpm.CDE_ID AS STRING)
    GROUP BY ms.segment
),
combined AS (
    SELECT
        sbs.segment,
        sbs.cost_bucket,
        sbs.signups,
        rbs.vouchers_redeemed
    FROM signups_by_segment sbs
    LEFT JOIN redeemed_by_segment rbs ON sbs.segment = rbs.segment
),
correction_factor AS (
    SELECT
        (SELECT SUM(redemption_count) FROM bf25_redemptions_per_member) * 1.0
        / (SELECT SUM(vouchers_redeemed) FROM combined) AS factor
),
cost_per_redemption AS (
    SELECT 6.37 AS cost  -- Patrick's Black Friday popcorn cost, stands as-is
),
bucket_costs AS (
    SELECT
        c.cost_bucket,
        SUM(c.signups) AS bucket_population,
        SUM(c.vouchers_redeemed * cf.factor) AS bucket_scaled_redemptions,
        SUM(c.vouchers_redeemed * cf.factor) * cpr.cost AS bucket_total_cost
    FROM combined c
    CROSS JOIN correction_factor cf
    CROSS JOIN cost_per_redemption cpr
    GROUP BY c.cost_bucket, cpr.cost
)
SELECT
    cost_bucket,
    bucket_population,
    ROUND(bucket_scaled_redemptions, 1) AS scaled_redemptions,
    ROUND(bucket_total_cost, 2) AS total_offer_cost,
    ROUND(bucket_total_cost / bucket_population, 2) AS cost_per_member
FROM bucket_costs
ORDER BY cost_bucket;

WITH
bf25_population AS (
    SELECT DISTINCT CAST(cl.cde_id AS STRING) AS cde_id
    FROM bf25_member_segments ms
    JOIN bf25_cde_lookup cl ON ms.cc_user_profile_id = cl.cc_user_profile_id
),

base_trans AS (
    SELECT
        t.CDE_ID AS cde_id,
        d.datefull,
        t.CineClubFLAG,
        t.SCENEFLAG,
        t.TicketFLAG,
        t.ConcessionFLAG,
        t.Quantity,
        t.Net_Revenue,
        t.CineClub_Discount_Type_ID AS discount_type_id,
        s.Auditorium_Experience_ID,
        s.Performance_Experience_ID,
        s.Is_DBOXflag
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s ON t.sessionskey = s.sessionskey
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.CDE_ID IS NOT NULL
      AND d.datefull BETWEEN DATE'2026-01-01' AND DATE'2026-08-01'
      AND (t.SCENEFLAG = 'Yes' OR t.CineClubFLAG = 'Yes')
),

segmented AS (
    SELECT
        bt.*,
        CASE
            WHEN bt.SCENEFLAG = 'Yes' AND bt.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 'Scene'
            WHEN bt.CineClubFLAG = 'Yes' AND CAST(bt.cde_id AS STRING) IN (SELECT cde_id FROM bf25_population) THEN 'CC_Acquired_Offer'
            WHEN bt.CineClubFLAG = 'Yes' THEN 'CC_Not_Acquired_Offer'
        END AS segment
    FROM base_trans bt
),

format_classified AS (
    SELECT
        *,
        CASE
            WHEN Auditorium_Experience_ID = 2      THEN 'AVX'
            WHEN Auditorium_Experience_ID = 3      THEN 'IMAX'
            WHEN Auditorium_Experience_ID = 4      THEN 'VIP'
            WHEN Auditorium_Experience_ID IN (5,6) THEN 'ALT'
            WHEN Auditorium_Experience_ID = 7      THEN 'FourDX'
            WHEN Auditorium_Experience_ID = 8      THEN 'ScreenX'
            WHEN Performance_Experience_ID = 2     THEN 'ThreeD'
            ELSE 'Regular'
        END AS final_format
    FROM segmented
),

visit_grain AS (
    SELECT
        segment,
        cde_id,
        datefull,
        SUM(CASE WHEN ConcessionFLAG = 'Yes' THEN Net_Revenue ELSE 0 END) AS concession_spend,
        CASE WHEN SUM(Quantity) > 0 THEN 1 ELSE 0 END AS is_visit
    FROM format_classified
    WHERE segment IS NOT NULL
    GROUP BY segment, cde_id, datefull
),

visit_agg AS (
    SELECT
        segment,
        COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END) AS distinct_members,
        SUM(is_visit) AS total_visits,
        ROUND(SUM(is_visit) * 1.0 / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_visits_per_member,
        ROUND(SUM(CASE WHEN is_visit = 1 THEN concession_spend ELSE 0 END) / COUNT(DISTINCT CASE WHEN is_visit = 1 THEN cde_id END), 2) AS avg_concession_spend_per_member
    FROM visit_grain
    GROUP BY segment
),

ticket_agg AS (
    SELECT
        segment,
        COUNT(DISTINCT cde_id) AS distinct_members_ticket,
        SUM(CASE WHEN discount_type_id = 6 THEN Quantity ELSE 0 END) AS tickets_redeemed,
        SUM(CASE WHEN discount_type_id = 5 THEN Quantity ELSE 0 END) AS tickets_companion,
        SUM(CASE WHEN final_format != 'Regular' OR Is_DBOXflag = 'Yes' THEN Quantity ELSE 0 END) AS tickets_premium
    FROM format_classified
    WHERE segment IS NOT NULL AND TicketFLAG = 'Yes'
    GROUP BY segment
)

SELECT
    va.segment,
    va.distinct_members,
    va.total_visits,
    va.avg_visits_per_member,
    ROUND(ta.tickets_redeemed * 1.0 / va.distinct_members, 2) AS tickets_redeemed_per_member,
    ROUND(ta.tickets_companion * 1.0 / va.distinct_members, 2) AS tickets_companion_per_member,
    va.avg_concession_spend_per_member,
    ROUND(ta.tickets_premium * 1.0 / va.distinct_members, 2) AS premium_tickets_per_member
FROM visit_agg va
LEFT JOIN ticket_agg ta ON va.segment = ta.segment
ORDER BY va.segment;

SELECT
    COUNT(DISTINCT bp.cde_id) AS bf25_population_total,
    COUNT(DISTINCT CASE WHEN snap.renewal_status LIKE 'Active%' THEN bp.cde_id END) AS still_active,
    COUNT(DISTINCT bp.cde_id) - COUNT(DISTINCT CASE WHEN snap.renewal_status LIKE 'Active%' THEN bp.cde_id END) AS churned
FROM bf25_population bp
LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot snap
    ON CAST(snap.cde_id AS STRING) = bp.cde_id
   AND snap.snapshot_date_id = '20260801'